In [2]:
#LibOQS for PQC algorithms from Opensafequantum.org can be installed per this method
import os
import sys

# Install dependencies
print("\n[1/6] Installing system dependencies...")
!apt-get update -qq
!apt-get install -y -qq cmake gcc g++ ninja-build libssl-dev git wget

# Build liboqs
print("\n[2/6] Building liboqs from source...")
!rm -rf /tmp/liboqs /usr/local/lib/liboqs.* /usr/local/include/oqs
!git clone --depth=1 https://github.com/open-quantum-safe/liboqs.git /tmp/liboqs
!cd /tmp/liboqs && mkdir build && cd build && cmake -GNinja -DCMAKE_INSTALL_PREFIX=/usr/local -DBUILD_SHARED_LIBS=ON .. && ninja && ninja install

print("\n[3/6] Setting library path...")
!ldconfig /usr/local/lib

# Verify library exists
print("\n[4/6] Verifying library installation...")
if os.path.exists('/usr/local/lib/liboqs.so'):
    print("✅ liboqs.so found at /usr/local/lib/liboqs.so")
else:
    print("❌ liboqs.so NOT FOUND")
    !ls -la /usr/local/lib/liboqs*

# Set environment variable BEFORE installing Python package
print("\n[5/6] Setting environment variables...")
os.environ['LD_LIBRARY_PATH'] = '/usr/local/lib:' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['OQS_INSTALL_DIR'] = '/usr/local'
!echo "export LD_LIBRARY_PATH=/usr/local/lib:$LD_LIBRARY_PATH" >> ~/.bashrc

print("\n[6/6] Installing Python packages...")
# Install liboqs-python and quantum computing frameworks
!pip install -q liboqs-python cirq qiskit qiskit-aer

print("\n" + "=" * 70)
print("✅ BASE INSTALLATION COMPLETE")
print("=" * 70)
print("\n⚠️  NOW RESTART RUNTIME: Runtime -> Restart Runtime")


[1/6] Installing system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../libssl-dev_3.0.2-0ubuntu1.29_amd64.deb ...
Unpacking libssl-dev:amd64 (3.0.2-0ubuntu1.29) over (3.0.2-0ubuntu1.26) ...
Preparing to unpack .../libssl3_3.0.2-0ubuntu1.29_amd64.deb ...
Unpacking libssl3:amd64 (3.0.2-0ubuntu1.29) over (3.0.2-0ubuntu1.26) ...
Setting up libssl3:amd64 (3.0.2-0ubuntu1.29) ...
Selecting previously unselected package ninja-build.
(Reading database ... 118419 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.10.1-1_amd64.deb ...
Unpacking ninja-build (1.10.1-1) ...
Setting up ninja-build (1.10.1-1) ...
Setting up libssl-dev:amd64 (3.0.2-0ubuntu1.29) ...
Processing triggers for man-

In [3]:
#Generate a legacy key pair to secure with PQC with the built in python library
from cryptography.fernet import Fernet

# Run this Function to generate and save the key
def generate_key():
  key = Fernet.generate_key()
  #Generate the key and save as a key file
  with open("secret.key", "wb") as key_file:
    key_file.write(key)
#Create the key
generate_key()






In [7]:
import oqs

In [5]:
# Use the original message that was to be encrypted
#Assume that  key was created before the advent of qpu threats with this libary
import os
from cryptography.fernet import Fernet

# Load the key from the key file
def load_key():
    return open("secret.key", "rb").read()

# Encrypt the file using the loaded key
def encrypt_file(file_name):
  # load the key and create an encrypted object
  key = load_key()
  fernet = Fernet(key)

  with open(file_name, "rb") as file:
    #Read the file and encrypt it
    file_data = file.read()
    encrypted_data = fernet.encrypt(file_data) # Corrected: call on instance

  #save the data for decryption
  with open(f"{file_name}.encrypted", "wb") as encrypted_file:
    encrypted_file.write(encrypted_data)

# Call the function to encrypt the data.txt file
encrypt_file('data.txt')

In [9]:
# Example: Use Hybrid key encapsulation with an approved PQC algorithm
import oqs
import time

def load_classical_key(filename):
    """Loads the classical key from the specified file as bytes."""
    with open(filename, 'rb') as f:
        return f.read()

def composite_key(classical_key_filename):
  #Use the NIST MLKEM-512 to generate the key
  pq_kem = oqs.KeyEncapsulation("ML-KEM-512")

  #determine the latency for PQC Key formation
  start = time.time()
  pq_public = pq_kem.generate_keypair()


  # Load the classical key as bytes from the file
  classical_key_bytes = load_classical_key(classical_key_filename)

  # Concatenate the byte arrays
  hybrid_public = classical_key_bytes + pq_public
  return hybrid_public

#Generate a  it with the legacy key
hybrid_file = composite_key('secret.key')

print("Public key size:", len(hybrid_file), "bytes")



Public key size: 844 bytes
